In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob

# --- 1. Configuration ---

# Define the single path to your baseline model's results
BASELINE_CSV_PATH = '/kaggle/input/r-b-aabb-w-yolov8n-s/yolo_training_results/yolov8s/results.csv'

# Define the output directory for the plot
OUTPUT_PLOT_DIRECTORY = "/kaggle/working/comparison_plots"
os.makedirs(OUTPUT_PLOT_DIRECTORY, exist_ok=True)

print("✅ Configuration loaded.")

# --- 2. Automatic Model Discovery ---
print("\n🔍 Starting automatic model discovery...")
# Initialize with the baseline model
MODELS_TO_COMPARE = {'Baseline': BASELINE_CSV_PATH}

try:
    # --- THIS SECTION HAS BEEN CORRECTED ---
    # Get the top-level directory of the baseline to exclude it from the search.
    # For '/kaggle/input/r-b-obb-w-yolo11n-s/...', this will be 'r-b-obb-w-yolo11n-s'.
    baseline_root_dir = BASELINE_CSV_PATH.split('/')[3]
    print(f"  - Baseline root directory to exclude: '{baseline_root_dir}'")

    # Use a flexible search pattern
    search_pattern = '/kaggle/input/*/yolo_training_results/*/results.csv'
    all_results_files = glob.glob(search_pattern)

    # Filter out any results that are inside the baseline's own root directory.
    comparison_files = [
        f for f in all_results_files if baseline_root_dir not in f.split('/')
    ]

    # Find the first available comparison file and label it "Augs."
    if comparison_files:
        augs_path = comparison_files[0]
        MODELS_TO_COMPARE['Augs.'] = augs_path
        print(f"  - Found comparison model and labeled it 'Augs.': {augs_path}")
    else:
        print("  - ⚠️ WARNING: No comparison models found in other directories.")

except Exception as e:
    print(f"  - ❌ ERROR during model discovery: {e}")

print(f"\n✅ Discovery complete. Models to compare: {list(MODELS_TO_COMPARE.keys())}")


# Define the metrics to plot from the CSV files
METRICS_TO_PLOT = {
    'mAP@0.5': 'metrics/mAP50(B)',
    'mAP@0.5:0.95': 'metrics/mAP50-95(B)',
    'Precision': 'metrics/precision(B)',
    'Recall': 'metrics/recall(B)'
}

# --- 3. Data Loading ---

loaded_data = {}
for name, path in MODELS_TO_COMPARE.items():
    try:
        if os.path.exists(path):
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            loaded_data[name] = df
            print(f"  - Successfully loaded data for '{name}'")
        else:
            print(f"  - ⚠️ WARNING: File not found for '{name}' at path: {path}")
    except Exception as e:
        print(f"  - ❌ ERROR: Could not load or process file for '{name}'. Reason: {e}")

if not loaded_data or len(loaded_data) < 2:
    print("\n❌ FATAL: A baseline and at least one comparison model must be loaded. Check paths.")
else:
    # --- 4. Plotting Logic ---
    print("\n🖼️  Generating comparison plot...")
    try:
        num_metrics = len(METRICS_TO_PLOT)
        fig, axes = plt.subplots(num_metrics, 1, figsize=(14, 7 * num_metrics), sharex=True)
        fig.suptitle('Baseline vs. Augmentation Performance', fontsize=20, weight='bold')

        for i, (title, column) in enumerate(METRICS_TO_PLOT.items()):
            ax = axes[i]
            
            # Plot Baseline as a dashed black line with the label "Baseline"
            if 'Baseline' in loaded_data:
                df_base = loaded_data['Baseline']
                if column in df_base.columns:
                    ax.plot(df_base['epoch'], df_base[column], color='black', linestyle='--', label='Baseline', linewidth=2.5)

            # Plot the other model as a solid blue line with the label "Augs."
            if 'Augs.' in loaded_data:
                df_aug = loaded_data['Augs.']
                if column in df_aug.columns:
                    ax.plot(df_aug['epoch'], df_aug[column], color='blue', label='Augs.', linewidth=2)

            ax.set_title(title, fontsize=14)
            ax.set_ylabel('Score')
            ax.grid(True, linestyle=':', alpha=0.7)
            ax.legend()

        axes[-1].set_xlabel('Epoch', fontsize=12)
        plt.tight_layout(rect=[0, 0.03, 1, 0.96])
        output_path = os.path.join(OUTPUT_PLOT_DIRECTORY, "Comparison_Baseline_vs_Augs.png")
        plt.savefig(output_path)
        plt.close(fig)

        print(f"\n✅ Plot successfully saved to: {output_path}")

    except Exception as e:
        print(f"\n❌ ERROR: An unexpected error occurred during plotting: {e}")

✅ Configuration loaded.

🔍 Starting automatic model discovery...
  - Baseline root directory to exclude: 'r-b-aabb-w-yolov8n-s'
  - Found comparison model and labeled it 'Augs.': /kaggle/input/augs-r-b-aabb-w-yolov8s/yolo_training_results/yolov8s/results.csv

✅ Discovery complete. Models to compare: ['Baseline', 'Augs.']
  - Successfully loaded data for 'Baseline'
  - Successfully loaded data for 'Augs.'

🖼️  Generating comparison plot...

✅ Plot successfully saved to: /kaggle/working/comparison_plots/Comparison_Baseline_vs_Augs.png
